# Human-in-the-Loop: Các cổng trước hành động, Phân tầng rủi ro và Ghi nhật ký kiểm toán

README cho bài học này giới thiệu Human-in-the-Loop bằng một đoạn ngắn yêu cầu người dùng `APPROVE` hoặc `REJECT` sau khi tác nhân đã tạo ra phản hồi. Mẫu này là một điểm khởi đầu tốt, nhưng các triển khai HITL trong thực tế thường cần thêm ba phần:

1. Một **cổng trước hành động** chạy **trước** khi tác nhân thực hiện bước có rủi ro, để chi phí, tính không thể đảo ngược và độ trễ được kiểm soát.
2. **Phân tầng rủi ro**, để các hành động rủi ro thấp tự động thực thi, hành động rủi ro trung bình được phê duyệt theo lô, và chỉ có hành động rủi ro cao mới chờ con người.
3. Một **nhật ký kiểm toán cộng với vòng lặp chỉnh sửa**, để mỗi quyết định cửa được ghi lại dưới dạng JSONL, và từ chối sẽ yêu cầu tác nhân thực hiện lại với lý do cấu trúc thay vì chỉ in `Revising...`.

Notebook này xây dựng từng phần trên cùng các nguyên thủy như `06-system-message-framework.ipynb`. Nó chạy từ đầu đến cuối trong chế độ `DEMO_MODE = True` (không cần nhập tương tác) hoặc với các lời nhắc `input()` thực khi `DEMO_MODE = False`. Lưu ý: trong DEMO_MODE, việc thử lại cho mục tiêu thứ ba được kịch bản hóa nên cơ chế vòng lặp được nhìn thấy rõ ràng. Việc phân loại lại dựa trên chỉnh sửa thực sự yêu cầu `DEMO_MODE = False` và một người điều hành.

**Ngoài phạm vi (xử lý trong các bài học khác):** xác thực và kiểm soát truy cập (mối đe dọa 2 trong README bài 06), middleware gọi công cụ (bài 14 phân tích MAF sâu), các mẫu tranh luận đa tác nhân.


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## Mẫu 1: Cửa kiểm soát trước hành động

Đoạn mã HITL trong README gọi tác nhân trước, sau đó hỏi người dùng để phê duyệt kết quả. Đó là luồng **sau hành động**. Tác nhân đã thực thi, vì vậy chi phí gọi LLM đã được trả, và bất kỳ tác động phụ nào (gửi email, ghi dòng cơ sở dữ liệu, bình luận bài đăng) đã xảy ra.

Luồng **trước hành động** chèn cửa kiểm soát trước khi tác nhân chạy bước có rủi ro. Tác nhân đề xuất hành động, cửa kiểm soát quyết định có thực thi hay không, và chỉ khi được phê duyệt thì tác động phụ mới xảy ra.

| Khía cạnh | Phê duyệt sau hành động (đoạn mã README) | Cửa kiểm soát trước hành động (sổ tay này) |
|---|---|---|
| Khi nào phê duyệt được chạy? | Sau khi tác nhân đã tạo ra kết quả | Trước khi bất kỳ tác động phụ nào được thực thi |
| Chi phí LLM khi bị từ chối | Đã trả | Chỉ trả cho đề xuất, không trả cho hành động |
| Tác động phụ không thể đảo ngược | Có thể xảy ra (hành động đã xảy ra) | Ngăn chặn được |
| Độ rõ ràng khi kiểm toán | Phê duyệt là câu lệnh in | Phê duyệt là bản ghi JSON với dấu thời gian, hành động, lý do |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Mô hình 2: Phân cấp rủi ro

Không phải mọi hành động đều cần sự phê duyệt của con người. Việc tra cứu chỉ đọc qua một API công khai có mức độ rủi ro khác với việc gửi email cho khách hàng. Đối xử cả hai như nhau là lãng phí sự chú ý của người vận hành và làm chậm tác nhân.

Một mô hình 3 cấp đơn giản:

| Cấp | Ví dụ | Quy trình phê duyệt |
|---|---|---|
| `thấp` (chỉ đọc) | Tìm kiếm cơ sở kiến thức, tra cứu lựa chọn chuyến bay, lấy trang web công khai | Tự động thực thi, ghi nhật ký để kiểm tra |
| `trung bình` (biến đổi đơn giản) | Lưu kết quả vào bộ nhớ đệm, tăng bộ đếm, lên lịch nhắc nhở | Tự động thực thi, nhưng xem lại hàng ngày theo lô |
| `cao` (giao tiếp bên ngoài hoặc không thể đảo ngược) | Gửi email, thanh toán thẻ, đăng lên kênh công khai | Chặn chờ phê duyệt của con người |

Đây là một cách phân cấp. Hệ thống sản xuất thường sử dụng các cấp độ chi tiết hơn (ví dụ: các cấp độ quyền IAM của AWS, phân cấp truy cập theo vai trò). Phiên bản 3 cấp dưới đây là phiên bản nhỏ nhất có ích cho một tác nhân kết hợp hành động chỉ đọc và có tác dụng phụ.

Bộ phân loại bên dưới sử dụng phương pháp heuristic từ khóa để bản demo giữ được tính xác định và rẻ tiền. Trong hệ thống sản xuất bạn sẽ thay thế bằng bộ phân loại học được hoặc bộ máy chính sách.


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Mẫu 3: Nhật ký kiểm tra và vòng lặp sửa đổi

Một `print("Response approved.")` không phải là nhật ký kiểm tra. Để tạo sự tin cậy, mọi quyết định qua cửa nên được ghi lại dưới dạng một sự kiện có cấu trúc mà bạn có thể truy vấn, phát lại hoặc đính kèm vào đánh giá sự cố sau này.

Hai phần:

1. **JSONL chỉ thêm mới.** Một dòng cho mỗi quyết định, kèm theo dấu thời gian, hành động, bậc, quyết định, lý do. Dễ grep, dễ gửi đến một kho lưu trữ nhật ký thực sự sau này.
2. **Vòng lặp sửa đổi khi bị từ chối.** Khi cửa trả về `deny`, tác nhân sẽ yêu cầu lại chính nó với lý do từ chối trong ngữ cảnh, để đề xuất tiếp theo có thể tránh vấn đề.


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Tài nguyên bổ sung

Một số dự án công khai khác triển khai các biến thể của các mô hình HITL này. So sánh các phương pháp để tìm ra cái phù hợp với stack của bạn:

- **LangChain** các lớp bao bọc công cụ human-in-the-loop ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): các lớp bao bọc công cụ có thể dừng thực thi để nhận đầu vào từ con người.
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ đã tái cấu trúc điều này): sử dụng vai trò tác nhân để đại diện cụ thể cho con người trong các cuộc trò chuyện đa tác nhân.
- **Microsoft Agent Framework (MAF)** middleware gọi hàm ([docs](https://learn.microsoft.com/agent-framework/)): middleware chạy xung quanh mỗi lần gọi công cụ/hàm, thích hợp cho logic kiểm soát và quy trình phê duyệt.

Mỗi dự án xử lý ba mô hình con khác nhau: LangChain bao bọc chúng như các công cụ, AutoGen sử dụng vai trò tác nhân, và Microsoft Agent Framework sử dụng middleware gọi hàm. Đọc một hoặc hai triển khai toàn diện trước khi chọn thiết kế cho tác nhân của bạn.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố miễn trừ trách nhiệm**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sai sót. Tài liệu gốc bằng ngôn ngữ gốc nên được coi là nguồn tin chính thức. Đối với thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp bởi con người. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
